In [ ]:
!pip -q install pyyaml pandas

In [ ]:
from pathlib import Path
import yaml, pandas as pd

for p in ['project/src','project/configs','project/outputs/logs','project/outputs/metrics','project/outputs/tables']:
    Path(p).mkdir(parents=True, exist_ok=True)
print('folders ready')

In [ ]:
from src.schemas import AgentConfig
from src.runner import run_variant
from src.eval import save_metrics
from src.io_utils import ensure_dirs, write_tables, save_config_snapshot

cfg = yaml.safe_load(open('project/configs/demo.yaml'))
ensure_dirs('project/outputs')
save_config_snapshot(cfg, 'project/outputs/config_snapshot.yaml')
print(cfg)

In [ ]:
from pathlib import Path
metrics_rows=[]

for scenario in cfg['scenarios']:
    for seed in cfg['seeds']:
        for variant in cfg['variants']:
            c = AgentConfig()
            if variant == 'no_tom': c.tom_enabled=False
            if variant == 'no_cf': c.cf_enabled=False
            if variant == 'no_profiler': c.profiler_enabled=False
            if variant == 'no_policies': c.policies_enabled=False
            log_path=f'project/outputs/logs/episode_{scenario}_{seed}_{variant}.jsonl'
            m=run_variant(scenario, seed, cfg['n_turns'], variant, c, log_path)
            m.update({'scenario_id':scenario,'seed':seed,'variant':variant})
            metrics_rows.append(m)

save_metrics(metrics_rows, 'project/outputs/metrics/metrics.csv')
write_tables('project/outputs/metrics/metrics.csv', 'project/outputs/tables')
print('done')

In [ ]:
print('metrics:', 'project/outputs/metrics/metrics.csv')
print('main table:', 'project/outputs/tables/table_main.tex')
print('ablation table:', 'project/outputs/tables/table_ablations.tex')
print(pd.read_csv('project/outputs/metrics/metrics.csv').head())

In [ ]:
from itertools import islice
log_file = sorted(Path('project/outputs/logs').glob('*.jsonl'))[0]
print('sample log:', log_file)
with open(log_file,'r',encoding='utf-8') as f:
    for line in islice(f,3):
        print(line.strip())